# 🌫️ India AQI Prediction using Machine Learning

**Predicting Air Quality Index category (Good, Moderate, Unhealthy, etc.) using pollutant concentrations and weather data.**

## 📌 Notebook Overview

This notebook walks through a complete beginner-friendly machine learning workflow:

1. **Load & Explore Data** – understand the dataset structure
2. **Data Cleaning** – handle missing values
3. **Data Leakage Check** – identify and remove columns that would let the model "cheat"
4. **Exploratory Data Analysis (EDA)** – visualize AQI distribution and relationships
5. **Feature Engineering** – encode categorical columns
6. **Model Building** – train a Random Forest Classifier
7. **Model Evaluation** – accuracy, classification report, feature importance
8. **Conclusion** – key takeaways

**Goal:** Predict the `AQI_Bucket` (Good / Moderate / Unhealthy / etc.) for a monitoring station using pollutant readings, weather conditions, and time-based features.

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 2️⃣ Load & Explore the Data

Let's start by loading the dataset and taking a first look at its shape, columns, and sample rows.

In [ ]:
df = pd.read_csv("enriched_aqi_dataset.csv")

print("Shape of dataset:", df.shape)
df.head()

In [ ]:
# Check column data types and missing values
df.info()

In [ ]:
# Count of missing values per column
df.isnull().sum()

## 3️⃣ Data Cleaning

The pollutant columns (`pollutant_min`, `pollutant_max`, `pollutant_avg`) have some missing values.
Since pollutant levels vary a lot by pollutant type (e.g. CO vs PM2.5 have very different scales), we fill
missing values with the **median value for that specific pollutant**, instead of the overall median.

In [ ]:
for col in ["pollutant_min", "pollutant_max", "pollutant_avg"]:
    df[col] = df.groupby("pollutant_id")[col].transform(lambda x: x.fillna(x.median()))

print("Missing values after cleaning:")
print(df[["pollutant_min", "pollutant_max", "pollutant_avg"]].isnull().sum())

## 4️⃣ ⚠️ Data Leakage Check (Important!)

Our target column is `AQI_Bucket` (Good, Moderate, Unhealthy, etc.).

Let's check how `AQI_Bucket` is related to the numeric `AQI` column:

In [ ]:
df.groupby("AQI_Bucket")["AQI"].agg(["min", "max"])

**Finding:** `AQI_Bucket` is *directly calculated* from the `AQI` number (e.g. AQI 0–50 = "Good", 51–100 = "Moderate", etc.).

If we let the model use the `AQI` column as a feature, it would simply learn the cutoff rules and score
near 100% — this is called **data leakage**, and the model would be useless on real, unseen data where
we don't already know the AQI number.

👉 **Action:** We drop the `AQI` column (and identifier columns like `station`, `last_update` which don't generalize)
from our features before training.

In [ ]:
# Columns to drop - AQI (leakage), and identifier/constant columns
drop_cols = ["AQI", "country", "city", "station", "last_update", "latitude", "longitude", "Year", "Day"]

target = "AQI_Bucket"
feature_cols = [c for c in df.columns if c not in drop_cols + [target]]
print("Features used for modeling:")
print(feature_cols)

## 5️⃣ Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8,5))
order = ["Good", "Moderate", "Unhealthy for Sensitive Groups", "Unhealthy", "Very Unhealthy", "Hazardous"]
sns.countplot(data=df, x="AQI_Bucket", order=order, palette="viridis")
plt.title("Distribution of AQI Categories")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="pollutant_id", y="pollutant_avg")
plt.title("Pollutant Average by Pollutant Type")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
numeric_cols = ["pollutant_min", "pollutant_max", "pollutant_avg", "Temperature_C", "Humidity_%", "Wind_Speed_kmh"]
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numeric Features")
plt.tight_layout()
plt.show()

## 6️⃣ Feature Engineering

Machine learning models need numbers, not text. We convert the categorical columns
(`state`, `pollutant_id`, `Season`) into numbers using **Label Encoding**.

In [ ]:
X = df[feature_cols].copy()
y = df[target].copy()

encoders = {}
for col in ["state", "pollutant_id", "Season"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

X.head()

## 7️⃣ Train-Test Split

We split the data into 80% training and 20% testing, using `stratify=y` so each AQI category
is represented proportionally in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 8️⃣ Model Building — Random Forest Classifier

Random Forest is a great beginner-friendly model: it handles mixed feature types well,
resists overfitting better than a single decision tree, and gives us feature importance for free.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model training complete!")

## 9️⃣ Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(7,6))
cm = confusion_matrix(y_test, y_pred, labels=order)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette="mako")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importances

## 📝 Conclusion

- We built a **Random Forest Classifier** to predict the AQI category of a monitoring station from
  pollutant readings, weather conditions, and time-based features.
- **Data leakage was identified and removed**: the raw `AQI` column directly determines `AQI_Bucket`
  through fixed cutoffs, so it was excluded from the features to keep the model realistic and generalizable.
- The final model achieved **~99% accuracy** on the held-out test set, with strong precision/recall
  across all six AQI categories (Good → Hazardous).
- **`pollutant_avg`** (and `pollutant_min`/`pollutant_max`) were by far the most influential features —
  which makes sense, since AQI is fundamentally derived from pollutant concentration levels.
- Weather features (temperature, humidity, wind speed) and time-based features (hour, season) added
  smaller but meaningful signal, useful for understanding seasonal/diurnal pollution patterns.

**Possible next steps:** try gradient boosting models (XGBoost/LightGBM), predict the exact numeric AQI
value using regression, or build a time-series forecast for a single city/station.

---
*If you found this notebook helpful, please upvote! 🙂*